# Crude export corridor EDA

Historical Kpler **Delivered Crude** liftings from a chosen origin corridor.

**Toggles**
- `CORRIDOR`: `BRAZIL` | `WAF` | `MEG` | `USG`
- `FLOW_SCOPE`: `INTER` (cross-border, default) | `INTRA` | `ALL`

**Fleet note:** §4 ranks the **mother** vessel (IMO 1). On STS trades the **daughter** (IMO 2) often delivers to the final destination — see §4b–4c.

Source: `data/kpler_trades.duckdb`.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from corridor_eda import (
    CORRIDORS,
    FLOW_SCOPES,
    apply_flow_scope,
    compare_flow_scopes,
    connect,
    daughter_owner_rank,
    daughter_recurrence,
    daughter_size_mix,
    daughter_size_mix_monthly,
    daughter_size_mix_monthly_tanker_count,
    direct_vlcc_installations,
    sts_vlcc_daughter_installations,
    vlcc_load_path_summary,
    dest_mix,
    dest_mix_monthly,
    fleet_concentration,
    flow_scope_summary,
    installation_rank,
    load_crude_exports,
    loading_cadence,
    loyalty_curve,
    monthly_export_mbd,
    mother_daughter_pairs,
    mother_role_features,
    mother_role_summary,
    owner_rank,
    size_mix,
    size_mix_monthly,
    size_mix_monthly_tanker_count,
    daughter_month_repeat_tankers,
    daughter_month_cargoes,
    mother_month_repeat_tankers,
    mother_month_cargoes,
    sts_delivery_summary,
    sts_share,
    vessel_recurrence,
    weekday_mix,
)

DB_PATH = ROOT / "data" / "kpler_trades.duckdb"
assert DB_PATH.exists(), f"Missing {DB_PATH} — run bootstrap_duckdb.py"

# --- toggles -----------------------------------------------------------------
CORRIDOR = "MEG"  # BRAZIL | WAF | MEG | USG
FLOW_SCOPE = "INTER"  # INTER | INTRA | ALL
INSPECT_MONTH = "2025-08-01"  # origin-month drill-down (YYYY-MM-01)
STATUS = "Delivered"
PRODUCT = "Crude"
START = "2018-01-01"
END = None
TOP_DEST_TRACK = None

pd.options.display.float_format = lambda x: f"{x:,.2f}"
cdef = CORRIDORS[CORRIDOR]
assert FLOW_SCOPE in FLOW_SCOPES, FLOW_SCOPES
display(Markdown(
    f"### Corridor: **{cdef.label}** (`{cdef.key}`) · scope: **{FLOW_SCOPE}**\n"
    f"{cdef.notes}\n\nOrigin filter: `{cdef.where_sql}`"
))


### Corridor: **Mideast Gulf** (`MEG`) · scope: **INTER**
Excludes Other Middle East (e.g. Yanbu, Fujairah) by design.

Origin filter: `"TradingRegion (origin)" = 'Mideast Gulf'`

In [2]:
con = connect(DB_PATH)
df_all = load_crude_exports(con, CORRIDOR, status=STATUS, product=PRODUCT, start=START, end=END)
df = apply_flow_scope(df_all, FLOW_SCOPE)
display(Markdown(
    f"Loaded **{len(df_all):,}** rows → scope `{FLOW_SCOPE}` keeps **{len(df):,}** | "
    f"trades: **{df['trade_id'].nunique():,}** | mother IMOs: **{df['imo'].nunique():,}** | "
    f"daughter IMOs: **{df['imo_2'].nunique():,}** | "
    f"volume: **{df['volume_bbl'].sum()/1e9:,.2f} Bbbl**"
))
display(df.head(3))

Loaded **58,430** rows → scope `INTER` keeps **55,297** | trades: **47,342** | mother IMOs: **1,898** | daughter IMOs: **964** | volume: **43.65 Bbbl**

,trade_id,voyage_id,date_origin,date_dest,trade_status,country_origin,region_origin,zone_origin,installation_origin,origin,...,volume_sts_1,month,year,has_sts,has_daughter,dwt_bucket,dwt_bucket_2,corridor,flow_scope,same_trading_region
0,5918143,26251861,2018-12-31 18:05:00,2019-01-16 06:46:00,Delivered,Saudi Arabia,Mideast Gulf,Ras Tanura,Ras Tanura,Ras Tanura,...,NaN,2018-12-01,2018,False,False,Suezmax,unknown,MEG,INTER,False
1,5918143,26251861,2018-12-31 18:05:00,2019-01-16 06:46:00,Delivered,Saudi Arabia,Mideast Gulf,Ras Tanura,Ras Tanura,Ras Tanura,...,NaN,2018-12-01,2018,False,False,Suezmax,unknown,MEG,INTER,False
2,5880351,26223447,2018-12-31 14:51:00,2019-01-19 02:38:00,Delivered,Saudi Arabia,Mideast Gulf,Ras Tanura,Juaymah,Juaymah,...,NaN,2018-12-01,2018,False,False,VLCC+,unknown,MEG,INTER,False


## 0. Intra vs inter (always shown)

In [3]:
fs = flow_scope_summary(df_all)
display(fs)
fig = px.bar(
    fs, x="flow_scope", y=["share_vol", "share_cargoes"], barmode="group",
    title=f"{cdef.label} — volume vs cargo share by flow scope",
)
fig.update_layout(height=360, yaxis_tickformat=".0%")
fig.show()

cmp_scope = compare_flow_scopes(df_all)
display(Markdown("### Metrics: ALL vs INTER vs INTRA"))
display(cmp_scope.T)

m_inter = monthly_export_mbd(apply_flow_scope(df_all, "INTER")).assign(scope="INTER")
m_intra = monthly_export_mbd(apply_flow_scope(df_all, "INTRA")).assign(scope="INTRA")
m_both = pd.concat([m_inter, m_intra], ignore_index=True)
if len(m_both):
    fig2 = px.line(m_both, x="month", y="mbd", color="scope",
                   title=f"{cdef.label} — mbd INTER vs INTRA")
    fig2.update_layout(height=400)
    fig2.show()

,flow_scope,cargoes,volume_bbl,share_cargoes,share_vol
0,INTER,47342,"43,645,872,437.00",0.94,0.94
1,INTRA,2037,"1,733,593,122.00",0.04,0.04
2,UNKNOWN,862,"1,077,401,324.00",0.02,0.02


### Metrics: ALL vs INTER vs INTRA

flow_scope,ALL,INTER,INTRA
trades,"50,241.00","47,342.00","2,037.00"
unique_imo,"1,952.00","1,898.00",237.00
volume_bbl,"46,456,866,883.00","43,645,872,437.00","1,733,593,122.00"
mbd_mean,14.55,13.67,0.54
china_vol_share,0.27,0.29,0.00
vlcc_plus_vol_share,0.84,0.85,0.59
suezmax_vol_share,0.12,0.12,0.12
top10_imo_lift_share,0.03,0.03,0.50
one_off_imo_share,0.10,0.10,0.44
sts_vol_share,0.08,0.08,0.02


## 1. Export rate (mbd) — current `FLOW_SCOPE`

In [4]:
m = monthly_export_mbd(df)
display(m.tail(12))
fig = go.Figure()
fig.add_trace(go.Scatter(x=m["month"], y=m["mbd"], mode="lines", name="mbd"))
fig.add_trace(go.Bar(x=m["month"], y=m["cargoes"], name="cargoes", yaxis="y2", opacity=0.35))
fig.update_layout(
    title=f"{cdef.label} [{FLOW_SCOPE}] crude — mbd + cargo count",
    height=420,
    yaxis=dict(title="mbd"),
    yaxis2=dict(title="cargoes", overlaying="y", side="right", showgrid=False),
    legend=dict(orientation="h"),
)
fig.show()

ms = m.copy()
ms["year"] = ms["month"].dt.year
ms["cal_month"] = ms["month"].dt.month
fig_s = px.line(ms, x="cal_month", y="mbd", color="year",
                title=f"{cdef.label} [{FLOW_SCOPE}] — month-of-year seasonality (mbd)")
fig_s.update_layout(height=380, xaxis=dict(dtick=1))
fig_s.show()

,month,cargoes,volume_bbl,mbd
93,2025-10-01,463,"441,389,491.00",14.24
94,2025-11-01,500,"445,085,781.00",14.84
95,2025-12-01,494,"464,825,044.00",14.99
96,2026-01-01,472,"443,717,522.00",14.31
97,2026-02-01,460,"405,880,660.00",14.50
98,2026-03-01,57,"65,210,464.00",2.10
99,2026-04-01,39,"43,412,491.00",1.45
100,2026-05-01,20,"19,224,347.00",0.62
101,2026-06-01,93,"119,761,750.00",3.99
102,2026-07-01,106,"125,831,785.00",4.06


## 2. Destination mix

In [5]:
if FLOW_SCOPE == "INTRA":
    dz = (
        df.groupby("zone_dest", as_index=False)
        .agg(cargoes=("trade_id", "nunique"), volume_bbl=("volume_bbl", "sum"))
        .sort_values("volume_bbl", ascending=False)
    )
    dz["share_vol"] = dz["volume_bbl"] / dz["volume_bbl"].sum()
    display(Markdown("### Top discharge zones (INTRA)"))
    display(dz.head(15))
    fig = px.bar(dz.head(12), x="zone_dest", y="share_vol",
                 title=f"{cdef.label} INTRA — discharge zones")
    fig.update_layout(height=400, yaxis_tickformat=".0%")
    fig.show()
else:
    dm = dest_mix(df, top_n=15)
    display(dm)
    fig = px.bar(dm, x="country_dest", y="share_vol",
                 title=f"{cdef.label} [{FLOW_SCOPE}] — top destinations")
    fig.update_layout(height=400, yaxis_tickformat=".0%")
    fig.show()
    dm_m = dest_mix_monthly(df, countries=TOP_DEST_TRACK)
    fig2 = px.area(dm_m, x="month", y="share", color="dest_bucket",
                   title=f"{cdef.label} [{FLOW_SCOPE}] — destination share over time")
    fig2.update_layout(height=420, yaxis_tickformat=".0%")
    fig2.show()

,country_dest,cargoes,volume_bbl,share_vol
10,China,9619,"12,515,455,008.00",0.29
16,India,6569,"6,961,792,009.00",0.16
20,Japan,8371,"5,714,766,473.00",0.13
42,South Korea,4898,"4,891,115,070.00",0.11
53,United States,4268,"1,992,086,928.00",0.05
47,Taiwan,1394,"1,591,687,201.00",0.04
40,Singapore Republic,2187,"1,502,189,300.00",0.03
12,Egypt,835,"1,159,831,154.00",0.03
49,Thailand,1771,"1,096,240,108.00",0.03
23,Malaysia,841,"606,706,568.00",0.01


## 3. Load installations

In [6]:
inst = installation_rank(df, top_n=20)
display(inst)
fig = px.bar(
    inst.sort_values("volume_bbl"),
    x="volume_bbl", y="installation_origin", orientation="h",
    title=f"{cdef.label} [{FLOW_SCOPE}] — top load installations (bbl)",
)
fig.update_layout(height=520)
fig.show()

sts = sts_share(df)
display(Markdown(
    f"STS present on **{100*sts['sts_cargo_share']:.1f}%** of cargoes / "
    f"**{100*sts['sts_vol_share']:.1f}%** of volume."
))

,installation_origin,cargoes,volume_bbl,share_vol
14,Juaymah,10833,"11,797,069,563.00",0.27
0,Al Basrah,9599,"9,917,705,457.00",0.23
30,Ras Tanura,6744,"5,701,052,296.00",0.13
25,Mina Al-Ahmadi,4432,"4,881,350,337.00",0.11
37,Zirku,3054,"2,437,111,776.00",0.06
17,Kharg Island,1731,"2,129,898,538.00",0.05
4,Das Island,3037,"1,979,242,248.00",0.05
13,Jebel Dhanna,1795,"940,212,526.00",0.02
8,FSO Qatar Petroleum Al Shaheen I,920,"584,575,214.00",0.01
11,Halul Island,723,"474,302,570.00",0.01


STS present on **11.9%** of cargoes / **8.1%** of volume.

## 4. Mother fleet stickiness (`FLOW_SCOPE`)

Ranks **Vessel / IMO 1** only. On STS exports these are often shuttles — not the ocean-going deliverer.

In [7]:
conc = fleet_concentration(df)
display(conc.to_frame("value"))
if len(conc):
    display(Markdown(
        f"Top-10 mother IMOs = **{100*conc['top10_lift_share']:.1f}%** of lifts; "
        f"one-off = **{100*conc['one_off_imo_share']:.1f}%**; HHI = **{conc['hhi_imo']:.4f}**."
    ))

recur = vessel_recurrence(df, top_n=30)
display(Markdown("### Top mothers in current scope"))
display(recur.head(20))

loy = loyalty_curve(df)
if len(loy):
    fig = go.Figure(go.Scatter(x=loy["rank"], y=loy["cum_lift_share"], mode="lines"))
    fig.add_hline(y=0.5, line_dash="dot", annotation_text="50%")
    fig.update_layout(
        title=f"{cdef.label} [{FLOW_SCOPE}] — cumulative lift share by mother IMO",
        height=380, yaxis_tickformat=".0%",
    )
    fig.show()

,value
n_trades,"47,342.00"
n_unique_imo,"1,898.00"
lifts_per_imo_mean,24.94
lifts_per_imo_median,15.00
top10_lift_share,0.03
top25_lift_share,0.07
hhi_imo,0.00
one_off_imo_share,0.10


Top-10 mother IMOs = **3.2%** of lifts; one-off = **10.0%**; HHI = **0.0012**.

### Top mothers in current scope

,imo,vessel,lifts,volume_bbl,vessel_dwt,vessel_type,vessel_owner,first_lift,last_lift,share_lifts,share_vol
637,9336842,Shalamar,172,"74,750,606.00","105,315.00",LR2,PNSC,2018-01-04 21:57:00,2026-04-13 06:00:00,0.00,0.00
684,9360233,Bright Pioneer,156,"181,041,093.00","302,403.00",VLCC,NISSEN KAIUN CO LTD,2018-01-17 05:01:00,2026-02-21 12:39:00,0.00,0.00
387,9270555,Integrity Racer,156,"71,536,614.00","107,215.00",LR2,<NA>,2018-01-01 13:55:00,2025-03-04 15:38:00,0.00,0.00
1056,9513402,Kazusa,149,"87,531,582.00","310,406.00",VLCC,Mitsui,2018-01-10 20:00:00,2026-02-23 18:58:00,0.00,0.00
1125,9562673,Libra Trader,147,"93,633,045.00","310,339.00",VLCC,Mitsui,2018-01-12 21:00:00,2026-01-24 12:34:00,0.00,0.00
407,9277541,Swift Galaxy,146,"73,318,415.00","107,018.00",LR2,<NA>,2018-01-11 05:55:00,2024-06-25 16:48:00,0.00,0.00
1283,9662887,Eneos Spirit,145,"88,153,238.00","311,984.00",VLCC,JX Ocean,2018-01-02 05:55:00,2026-01-24 16:53:00,0.00,0.00
1254,9607863,Mitake,144,"89,824,952.00","305,219.00",VLCC,Mitsui,2018-01-09 13:49:00,2026-01-04 02:17:00,0.00,0.00
1574,9814117,Genei,141,"91,643,911.00","312,375.00",VLCC,NYK Mitsubishi JV,2018-04-22 14:32:00,2026-01-24 16:16:00,0.00,0.00
1127,9562697,Gassan,141,"91,863,650.00","308,209.00",VLCC,Mitsui,2018-01-25 21:30:00,2026-01-31 18:19:00,0.00,0.00


## 4b. Mother roles (from full corridor behaviour)

Roles are classified on **`df_all`** (INTER+INTRA) so a dual-use ship is not mislabeled by the current scope filter.

| Role | Rule |
|---|---|
| `STS_EXPORT` | ≥70% INTER and ≥70% STS |
| `COASTAL` | ≥70% INTRA and STS &lt;30% |
| `DUAL` | INTER and INTRA each ≥25% |
| `OTHER` | residual |

In [8]:
roles = mother_role_features(df_all, min_lifts=5)
role_sum = mother_role_summary(roles)
display(role_sum)

fig = px.bar(role_sum, x="role", y="share_vol", title=f"{cdef.label} — mother volume by role")
fig.update_layout(height=360, yaxis_tickformat=".0%")
fig.show()

if len(roles):
    fig_s = px.scatter(
        roles, x="pct_inter", y="pct_sts", color="role", size="lifts",
        hover_data=["vessel", "imo", "vessel_owner", "pct_intra"],
        title=f"{cdef.label} — mother roles (% INTER vs % STS)",
    )
    fig_s.update_layout(height=480, xaxis_tickformat=".0%", yaxis_tickformat=".0%")
    fig_s.show()

display(Markdown("### Top STS_EXPORT mothers"))
display(roles.loc[roles["role"] == "STS_EXPORT"].head(15))
display(Markdown("### Top COASTAL mothers"))
display(roles.loc[roles["role"] == "COASTAL"].head(15))
display(Markdown("### Top DUAL mothers"))
display(roles.loc[roles["role"] == "DUAL"].head(15))

# How do top scope mothers classify?
if len(recur):
    top_roles = recur.merge(roles[["imo", "role", "pct_inter", "pct_intra", "pct_sts"]], on="imo", how="left")
    display(Markdown("### Roles of top mothers in current `FLOW_SCOPE`"))
    display(top_roles[["imo", "vessel", "lifts", "role", "pct_inter", "pct_intra", "pct_sts"]].head(15))

,role,n_imo,lifts,volume_bbl,share_lifts,share_vol
2,OTHER,1370,45953,"43,251,388,648.00",0.93,0.95
0,COASTAL,31,1485,"1,211,377,315.00",0.03,0.03
3,STS_EXPORT,23,1200,"701,494,785.00",0.02,0.02
1,DUAL,18,532,"567,139,225.00",0.01,0.01


### Top STS_EXPORT mothers

,imo,vessel,lifts,volume_bbl,vessel_dwt,vessel_owner,n_inter,n_intra,n_sts,n_with_daughter,pct_inter,pct_intra,pct_sts,pct_with_daughter,role,dwt_bucket
1514,9779824,Aslaf,83,"55,779,264.00","298,778.00",Bahri,89,4,72,72,1.07,0.05,0.87,0.87,STS_EXPORT,VLCC+
1525,9783679,Khurais,70,"54,568,057.00","298,706.00",Bahri,71,6,53,53,1.01,0.09,0.76,0.76,STS_EXPORT,VLCC+
1528,9783708,Qamran,69,"49,083,074.00","298,858.00",Bahri,68,8,53,53,0.99,0.12,0.77,0.77,STS_EXPORT,VLCC+
1515,9779836,Rimthan,68,"45,936,741.00","298,855.00",Bahri,65,7,56,56,0.96,0.10,0.82,0.82,STS_EXPORT,VLCC+
1527,9783693,Amad,67,"52,602,070.00","298,705.00",Bahri,66,12,49,49,0.99,0.18,0.73,0.73,STS_EXPORT,VLCC+
1526,9783681,Lawhah,66,"46,588,015.00","298,794.00",Bahri,62,6,51,51,0.94,0.09,0.77,0.77,STS_EXPORT,VLCC+
1529,9783710,Kassab,62,"43,548,672.00","298,721.00",Bahri,70,4,55,55,1.13,0.06,0.89,0.89,STS_EXPORT,VLCC+
1687,9834430,San Ramon Voyager,62,"30,254,892.00","318,607.00",Maran Tankers Management Inc,71,0,46,46,1.15,0.00,0.74,0.74,STS_EXPORT,VLCC+
1219,9583732,Almi Hercules,61,"33,536,915.00","319,359.00",ALMI TANKERS SA,70,0,43,43,1.15,0.00,0.70,0.70,STS_EXPORT,VLCC+
1133,9528940,Dht Redwood,61,"30,883,539.00","318,129.00",DHT Norway,68,0,47,47,1.11,0.00,0.77,0.77,STS_EXPORT,VLCC+


### Top COASTAL mothers

,imo,vessel,lifts,volume_bbl,vessel_dwt,vessel_owner,n_inter,n_intra,n_sts,n_with_daughter,pct_inter,pct_intra,pct_sts,pct_with_daughter,role,dwt_bucket
118,9187631,Apama,218,"112,127,612.00","99,087.00",NITC,2,216,2,2,0.01,0.99,0.01,0.01,COASTAL,Aframax/LR2
119,9187643,Arnica,176,"82,779,193.00","99,148.00",NITC,2,173,0,0,0.01,0.98,0.00,0.00,COASTAL,Aframax/LR2
121,9187667,Argo I,160,"83,261,379.00","99,094.00",NITC,10,148,2,2,0.06,0.93,0.01,0.01,COASTAL,Aframax/LR2
117,9187629,Artavil,159,"83,840,761.00","99,144.00",NITC,0,158,0,0,0.00,0.99,0.00,0.00,COASTAL,Aframax/LR2
530,9297515,An Tai,104,"80,046,011.00","159,089.00",<NA>,13,92,0,0,0.12,0.88,0.00,0.00,COASTAL,Suezmax
463,9288887,Nordic Freedom,102,"79,618,042.00","159,331.00",<NA>,29,74,4,4,0.28,0.73,0.04,0.04,COASTAL,Suezmax
188,9220952,Quinn,54,"99,382,674.00","303,115.00",<NA>,21,51,2,2,0.39,0.94,0.04,0.04,COASTAL,VLCC+
120,9187655,Ark III,49,"26,234,957.00","99,030.00",NITC,2,47,0,0,0.04,0.96,0.00,0.00,COASTAL,Aframax/LR2
274,9237761,Tina 5,46,"79,874,815.00","317,000.00",<NA>,28,40,0,0,0.61,0.87,0.00,0.00,COASTAL,VLCC+
189,9220964,St3,45,"81,037,422.00","302,977.00",Bahri,24,43,2,2,0.53,0.96,0.04,0.04,COASTAL,VLCC+


### Top DUAL mothers

,imo,vessel,lifts,volume_bbl,vessel_dwt,vessel_owner,n_inter,n_intra,n_sts,n_with_daughter,pct_inter,pct_intra,pct_sts,pct_with_daughter,role,dwt_bucket
1058,9484728,Ghinah,70,"80,775,804.00","319,329.00",Bahri,75,21,28,28,1.07,0.30,0.40,0.40,DUAL,VLCC+
1116,9519717,Awtad,68,"60,931,038.00","309,988.00",Bahri,66,22,41,41,0.97,0.32,0.60,0.60,DUAL,VLCC+
1002,9448700,Dilam,56,"70,443,378.00","320,106.00",Bahri,52,24,19,19,0.93,0.43,0.34,0.34,DUAL,VLCC+
277,9237797,Tima,54,"71,648,546.00","317,000.00",<NA>,51,15,10,10,0.94,0.28,0.19,0.19,DUAL,VLCC+
656,9332535,Sea V,53,"62,125,879.00","317,788.00",Bahri,43,16,21,21,0.81,0.30,0.40,0.40,DUAL,VLCC+
981,9438418,Nordic Cross,48,"31,027,904.00","158,475.00",<NA>,15,33,1,1,0.31,0.69,0.02,0.02,DUAL,Suezmax
663,9336098,Layla,43,"46,249,928.00","317,821.00",Bahri,40,13,18,18,0.93,0.30,0.42,0.42,DUAL,VLCC+
896,9412581,Nordic Vega,31,"23,849,233.00","164,028.00",<NA>,10,21,0,0,0.32,0.68,0.00,0.00,DUAL,Suezmax
29,9102277,Itaugua,21,"36,514,077.00","300,361.00",<NA>,14,9,9,9,0.67,0.43,0.43,0.43,DUAL,VLCC+
26,9102241,Shelif,18,"30,526,884.00","300,361.00",<NA>,12,7,9,9,0.67,0.39,0.50,0.50,DUAL,VLCC+


### Roles of top mothers in current `FLOW_SCOPE`

,imo,vessel,lifts,role,pct_inter,pct_intra,pct_sts
0,9336842,Shalamar,172,OTHER,1.11,0.01,0.00
1,9360233,Bright Pioneer,156,OTHER,1.28,0.00,0.00
2,9270555,Integrity Racer,156,OTHER,1.06,0.00,0.00
3,9513402,Kazusa,149,OTHER,1.23,0.00,0.00
4,9562673,Libra Trader,147,OTHER,1.07,0.00,0.00
5,9277541,Swift Galaxy,146,OTHER,1.12,0.00,0.00
6,9662887,Eneos Spirit,145,OTHER,1.25,0.00,0.00
7,9607863,Mitake,144,OTHER,1.18,0.00,0.00
8,9814117,Genei,141,OTHER,1.06,0.00,0.00
9,9562697,Gassan,141,OTHER,1.21,0.00,0.01


## 4c. Daughter fleet (Vessel IMO 2) — final delivery

On STS rows, Kpler `Vessel 2` is typically the ocean-going ship after STS.
Also: **can origins load VLCCs directly**, or is VLCC-scale export almost always Suezmax→STS→VLCC?


In [9]:
deliv = sts_delivery_summary(df)
display(deliv.to_frame("value"))
display(Markdown(
    f"In `{FLOW_SCOPE}`: STS = **{100*deliv.get('sts_vol_share', float('nan')):.1f}%** vol; "
    f"with daughter = **{100*deliv.get('daughter_vol_share', float('nan')):.1f}%**; "
    f"daughter VLCC+ = **{100*deliv.get('daughter_vlcc_vol_share', float('nan')):.1f}%**; "
    f"unique daughters = **{int(deliv.get('n_unique_daughters', 0))}**."
))

dau = daughter_recurrence(df, top_n=30)
display(Markdown("### Top daughter IMOs (final delivery)"))
display(dau)

dsm = daughter_size_mix(df)
if len(dsm):
    fig = px.pie(dsm, names="dwt_bucket_2", values="volume_bbl",
                 title=f"{cdef.label} [{FLOW_SCOPE}] — daughter volume by DWT")
    fig.show()

# Daughter DWT share over time
dsm_m = daughter_size_mix_monthly(df)
if len(dsm_m):
    fig_t = px.area(
        dsm_m, x="month", y="share", color="dwt_bucket_2",
        title=f"{cdef.label} [{FLOW_SCOPE}] — daughter DWT share over time",
        category_orders={"dwt_bucket_2": ["VLCC+", "Suezmax", "Aframax/LR2", "Panamax/LR1", "smaller", "unknown"]},
    )
    fig_t.update_layout(height=400, yaxis_tickformat=".0%")
    fig_t.show()

# Daughter unique tanker counts over time (same DWT buckets)
dsm_c = daughter_size_mix_monthly_tanker_count(df)
if len(dsm_c):
    fig_tc = px.bar(
        dsm_c, x="month", y="n_tankers", color="dwt_bucket_2",
        title=f"{cdef.label} [{FLOW_SCOPE}] — daughter unique tankers by DWT",
        category_orders={"dwt_bucket_2": ["VLCC+", "Suezmax", "Aframax/LR2", "Panamax/LR1", "smaller", "unknown"]},
        hover_data=["cargoes"],
    )
    fig_tc.update_layout(height=400, barmode="stack", yaxis_title="unique IMOs")
    fig_tc.show()


# --- Inspect one origin month: why unique daughters < cargoes ---
display(Markdown(
    f"### Month inspect — daughters (`INSPECT_MONTH={INSPECT_MONTH}`)\n"
    "Same IMO can appear on multiple origin-month trades (part-cargos, short hauls, "
    "or STS chain timing). `voyage_days` / `gap_days` show whether Far East reload is plausible."
))
d_rep = daughter_month_repeat_tankers(df, INSPECT_MONTH)
d_cgo = daughter_month_cargoes(df, INSPECT_MONTH)
n_dau = int(d_cgo["imo_2"].nunique()) if len(d_cgo) else 0
n_cgo = int(d_cgo["trade_id"].nunique()) if len(d_cgo) else 0
display(Markdown(
    f"**{INSPECT_MONTH}:** unique daughters = **{n_dau}**, cargoes = **{n_cgo}**, "
    f"IMOs with >1 lift = **{len(d_rep)}**."
))
display(Markdown("Repeat daughter IMOs (>1 lift in month)"))
display(d_rep)
display(Markdown("All daughter cargoes that month (sorted by repeat lifts)"))
display(d_cgo)

down = daughter_owner_rank(df, top_n=20)
display(Markdown("### Top daughter owners"))
display(down)

pairs = mother_daughter_pairs(df, top_n=25)
display(Markdown("### Top mother → daughter pairs"))
display(pairs)

if len(dau):
    fig_d = px.bar(
        dau.head(15).sort_values("volume_bbl"),
        x="volume_bbl", y="vessel_2", orientation="h",
        hover_data=["imo_2", "n_mothers", "top_dest", "vessel_owner_2"],
        title=f"{cdef.label} [{FLOW_SCOPE}] — top daughters by volume",
    )
    fig_d.update_layout(height=480)
    fig_d.show()

# --- Direct VLCC load vs STS-fed VLCC daughter ---
display(Markdown(
    "### VLCC load path: direct at origin vs STS from Suezmax\n"
    "- `DIRECT_VLCC_MOTHER` — mother ≥200 kdwt, **no STS** (VLCC loaded at installation)\n"
    "- `STS_TO_VLCC_DAUGHTER` — daughter ≥200 kdwt after STS (typical shuttle pattern)\n"
    "- `OTHER` — export without a VLCC-scale leg in mother/daughter slots"
))
vpath = vlcc_load_path_summary(df)
display(vpath)
if len(vpath):
    fig_p = px.bar(
        vpath, x="vlcc_load_path", y="share_vol",
        title=f"{cdef.label} [{FLOW_SCOPE}] — volume by VLCC load path",
    )
    fig_p.update_layout(height=360, yaxis_tickformat=".0%")
    fig_p.show()

direct = direct_vlcc_installations(df, top_n=20)
display(Markdown("### Installations that load VLCCs **directly** (no STS)"))
display(direct)

sts_feed = sts_vlcc_daughter_installations(df, top_n=20)
display(Markdown("### Installations feeding **STS → VLCC daughter** (shuttle load points)"))
display(sts_feed)

if len(vpath):
    direct_share = float(vpath.loc[vpath["vlcc_load_path"] == "DIRECT_VLCC_MOTHER", "share_vol"].sum())
    sts_share_v = float(vpath.loc[vpath["vlcc_load_path"] == "STS_TO_VLCC_DAUGHTER", "share_vol"].sum())
    display(Markdown(
        f"**Takeaway:** direct VLCC mother = **{100*direct_share:.1f}%** of `{FLOW_SCOPE}` volume; "
        f"STS→VLCC daughter = **{100*sts_share_v:.1f}%**. "
        + (
            "Most VLCC-scale export is **not** loaded as a VLCC at the FPSO — it is shuttled then STS'd."
            if sts_share_v > direct_share
            else "Direct VLCC loads are material here."
        )
    ))


,value
trades,"47,342.00"
sts_vol_share,0.08
daughter_vol_share,0.08
daughter_vlcc_vol_share,0.03
n_unique_mothers_sts,841.00
n_unique_daughters,964.00


In `INTER`: STS = **8.1%** vol; with daughter = **8.1%**; daughter VLCC+ = **2.9%**; unique daughters = **964**.

### Top daughter IMOs (final delivery)

,imo_2,vessel_2,lifts,volume_bbl,vessel_dwt_2,vessel_type_2,vessel_owner_2,n_mothers,top_dest,top_sts_zone,first_lift,last_lift,share_lifts,share_vol,dwt_bucket_2
733,9665748,Polaris Voyager,517,"195,098,005.00","155,720.00",LR3,Chevron,92,United States,WCL Light.,2018-01-10 13:42:00,2026-02-24 20:39:00,0.08,0.06,Suezmax
732,9665736,Pegasus Voyager,505,"195,378,864.00","155,720.00",LR3,Chevron,96,United States,WCL Light.,2018-01-10 13:42:00,2026-02-24 20:39:00,0.08,0.06,Suezmax
418,9344813,Lian Yun Hu,210,"84,080,077.00","75,493.00",LR1,COSCO,67,China,Qinzhou Light.,2018-01-02 19:59:00,2021-12-10 06:58:00,0.03,0.02,Panamax/LR1
615,9453999,Eagle Hydra,187,"67,734,340.00","113,968.00",LR2,<NA>,58,United States,PAL Light.,2018-01-14 06:00:00,2025-12-31 15:18:00,0.03,0.02,Aframax/LR2
657,9518907,Eagle Texas,135,"62,982,208.00","107,481.00",LR2,MISC,68,United States,Galveston Light.,2018-01-06 08:08:00,2026-02-08 23:18:00,0.02,0.02,Aframax/LR2
419,9344825,Lian Xing Hu,124,"47,620,642.00","75,504.00",LR1,COSCO,49,China,Qinzhou Light.,2019-09-24 20:09:00,2021-06-11 16:00:00,0.02,0.01,Panamax/LR1
420,9344837,Lian Sheng Hu,123,"43,904,585.00","75,499.00",LR1,COSCO,39,China,Qinzhou Light.,2018-01-02 19:59:00,2021-12-10 06:58:00,0.02,0.01,Panamax/LR1
656,9518892,Eagle Louisiana,122,"56,068,212.00","102,926.00",LR2,MISC,58,United States,Galveston Light.,2018-01-07 10:24:00,2026-01-26 00:19:00,0.02,0.02,Aframax/LR2
556,9417892,Eagle Klang,93,"42,118,534.00","107,481.00",LR2,MISC,49,United States,Galveston Light.,2018-02-26 10:00:00,2025-12-05 09:18:00,0.02,0.01,Aframax/LR2
353,9312602,Tian Long Zuo,93,"37,041,909.00","69,996.00",LR1,COSCO,36,China,Qinzhou Light.,2019-02-26 20:52:00,2021-07-29 17:03:00,0.02,0.01,Panamax/LR1


### Month inspect — daughters (`INSPECT_MONTH=2025-08-01`)
Same IMO can appear on multiple origin-month trades (part-cargos, short hauls, or STS chain timing). `voyage_days` / `gap_days` show whether Far East reload is plausible.

...

### Installations that load VLCCs **directly** (no STS)

,installation_origin,cargoes,volume_bbl,n_imo,top_dest,share_vol
10,Juaymah,8078,"10,423,748,131.00",838,China,0.31
0,Al Basrah,5426,"6,944,750,436.00",742,China,0.21
17,Mina Al-Ahmadi,3923,"4,546,103,539.00",673,Japan,0.14
21,Ras Tanura,4249,"4,123,688,111.00",736,Japan,0.12
25,Zirku,2424,"2,034,357,448.00",677,China,0.06
2,Das Island,2160,"1,512,005,095.00",496,Japan,0.04
12,Kharg Island,515,"773,767,674.00",140,China,0.02
9,Jebel Dhanna,994,"535,868,897.00",413,Japan,0.02
5,FSO Qatar Petroleum Al Shaheen I,767,"509,333,458.00",386,Japan,0.02
8,Halul Island,615,"410,202,311.00",305,Japan,0.01


### Installations feeding **STS → VLCC daughter** (shuttle load points)

,installation_origin,cargoes,volume_bbl,n_mothers,n_daughters,top_sts,top_dest,share_vol
13,Kharg Island,492,"779,221,700.00",106,116,EOPL,China,0.64
0,Al Basrah,90,"108,699,325.00",51,57,Sohar Light.,China,0.09
24,Zirku,60,"65,555,451.00",30,36,Fujairah Light.,China,0.05
17,Mina Al-Ahmadi,53,"64,422,927.00",14,31,Fujairah Light.,China,0.05
1,Das Island,42,"42,075,792.00",16,23,Fujairah Light.,India,0.03
11,Juaymah,21,"26,448,383.00",17,18,Linggi Light.,China,0.02
18,Mina Saud,48,"20,572,036.00",27,33,Fujairah Light.,United States,0.02
3,FPSO Soroosh,23,"17,411,570.00",15,19,EOPL,China,0.01
22,Umm Lulu,16,"16,093,416.00",8,10,Fujairah Light.,Thailand,0.01
21,Sirri,16,"15,396,870.00",12,12,EOPL,China,0.01


**Takeaway:** direct VLCC mother = **77.5%** of `INTER` volume; STS→VLCC daughter = **2.9%**. Direct VLCC loads are material here.

## 5. Size mix (mother DWT buckets)

In [10]:
sm = size_mix(df)
display(sm)
fig = px.pie(sm, names="dwt_bucket", values="volume_bbl",
             title=f"{cdef.label} [{FLOW_SCOPE}] — mother volume by DWT")
fig.show()

sm_m = size_mix_monthly(df)
fig2 = px.area(
    sm_m, x="month", y="share", color="dwt_bucket",
    title=f"{cdef.label} [{FLOW_SCOPE}] — mother DWT share over time",
    category_orders={"dwt_bucket": ["VLCC+", "Suezmax", "Aframax/LR2", "Panamax/LR1", "smaller", "unknown"]},
)
fig2.update_layout(height=400, yaxis_tickformat=".0%")
fig2.show()

# Mother unique tanker counts over time (same DWT buckets)
sm_c = size_mix_monthly_tanker_count(df)
if len(sm_c):
    fig3 = px.bar(
        sm_c, x="month", y="n_tankers", color="dwt_bucket",
        title=f"{cdef.label} [{FLOW_SCOPE}] — mother unique tankers by DWT",
        category_orders={"dwt_bucket": ["VLCC+", "Suezmax", "Aframax/LR2", "Panamax/LR1", "smaller", "unknown"]},
        hover_data=["cargoes"],
    )
    fig3.update_layout(height=400, barmode="stack", yaxis_title="unique IMOs")
    fig3.show()


# --- Inspect one origin month: why unique mothers < cargoes ---
display(Markdown(
    f"### Month inspect — mothers (`INSPECT_MONTH={INSPECT_MONTH}`)\n"
    "Charts count **unique IMOs**; cargoes count **trades**. Repeat lifts on the same "
    "mother (esp. STS shuttles) drive `n_tankers < cargoes`."
))
m_rep = mother_month_repeat_tankers(df, INSPECT_MONTH)
m_cgo = mother_month_cargoes(df, INSPECT_MONTH)
n_mom = int(m_cgo["imo"].nunique()) if len(m_cgo) else 0
n_cgo = int(m_cgo["trade_id"].nunique()) if len(m_cgo) else 0
display(Markdown(
    f"**{INSPECT_MONTH}:** unique mothers = **{n_mom}**, cargoes = **{n_cgo}**, "
    f"IMOs with >1 lift = **{len(m_rep)}**."
))
display(Markdown("Repeat mother IMOs (>1 lift in month)"))
display(m_rep)
display(Markdown("All mother cargoes that month (sorted by repeat lifts)"))
display(m_cgo)


,dwt_bucket,cargoes,volume_bbl,share_vol
3,VLCC+,37841,"37,083,784,962.00",0.85
2,Suezmax,7162,"5,419,684,871.00",0.12
0,Aframax/LR2,2299,"1,128,229,734.00",0.03
1,Panamax/LR1,34,"12,637,920.00",0.00
4,smaller,6,"1,534,950.00",0.00


### Month inspect — mothers (`INSPECT_MONTH=2025-08-01`)
Charts count **unique IMOs**; cargoes count **trades**. Repeat lifts on the same mother (esp. STS shuttles) drive `n_tankers < cargoes`.

**2025-08-01:** unique mothers = **261**, cargoes = **469**, IMOs with >1 lift = **129**.

Repeat mother IMOs (>1 lift in month)

,imo,vessel,dwt_bucket,lifts,volume_bbl,first_origin,last_origin,n_dest,top_dest,median_voyage_days,n_sts,gap_days
202,9813448,Dht Stallion,VLCC+,6,"2,116,580.00",2025-08-26 08:58:00,2025-08-29 12:38:00,1,United States,51.00,6,3
65,9397145,Atlantic Pioneer,VLCC+,5,"3,568,357.00",2025-08-04 12:59:00,2025-08-24 07:38:00,1,India,7.00,0,19
209,9830800,Nafees,VLCC+,5,"1,879,324.00",2025-08-13 19:59:00,2025-08-13 19:59:00,1,United States,51.00,5,0
255,9930806,Maran Dione,VLCC+,5,"1,800,387.00",2025-08-10 04:58:00,2025-08-14 02:59:00,1,Thailand,21.00,3,3
22,9324100,Lila Vadinar,VLCC+,4,"4,166,997.00",2025-08-02 02:39:00,2025-08-30 07:59:00,1,India,9.00,0,28
...,...,...,...,...,...,...,...,...,...,...,...,...
45,9378864,Visund,Suezmax,2,"800,001.00",2025-08-20 19:59:00,2025-08-20 19:59:00,1,India,6.00,0,0
70,9402914,Maran Penelope,Suezmax,2,"700,785.00",2025-08-16 00:59:00,2025-08-16 00:59:00,1,United States,46.00,0,0
191,9787792,Bright Crest,VLCC+,2,"675,814.00",2025-08-30 00:39:00,2025-08-30 00:39:00,1,Japan,31.00,0,0
39,9360453,Mardan,Aframax/LR2,2,"675,241.00",2025-08-13 05:39:00,2025-08-26 09:57:00,1,Pakistan,4.50,0,13


All mother cargoes that month (sorted by repeat lifts)

,date_origin,date_dest,voyage_days,trade_id,vessel,imo,dwt_bucket,vessel_dwt,volume_bbl,installation_origin,destination,country_dest,region_dest,has_sts,zone_sts_1,vessel_2,imo_2,dwt_bucket_2,imo_lifts_in_month,grade
52322,2025-08-26 08:58:00,2025-10-20 19:19:00,55,29790858,Dht Stallion,9813448,VLCC+,"319,713.00","317,371.00",Al Basrah,Chevron Richmond,United States,USWC (US West Coast),True,WCL Light.,Polaris Voyager,9665748,Suezmax,6,Basrah Med.
52323,2025-08-26 08:58:00,2025-10-19 08:19:00,53,29790859,Dht Stallion,9813448,VLCC+,"319,713.00","319,352.00",Al Basrah,Chevron Richmond,United States,USWC (US West Coast),True,WCL Light.,Pegasus Voyager,9665736,Suezmax,6,Basrah Med.
52324,2025-08-26 08:58:00,2025-10-13 01:59:00,47,29790860,Dht Stallion,9813448,VLCC+,"319,713.00","319,352.00",Al Basrah,Chevron Richmond,United States,USWC (US West Coast),True,WCL Light.,Polaris Voyager,9665748,Suezmax,6,Basrah Med.
52264,2025-08-29 12:38:00,2025-10-20 19:19:00,52,29790891,Dht Stallion,9813448,VLCC+,"319,713.00","388,873.00",Juaymah,Chevron Richmond,United States,USWC (US West Coast),True,WCL Light.,Polaris Voyager,9665748,Suezmax,6,Arab Lt.
52265,2025-08-29 12:38:00,2025-10-19 08:19:00,50,29790893,Dht Stallion,9813448,VLCC+,"319,713.00","385,816.00",Juaymah,Chevron Richmond,United States,USWC (US West Coast),True,WCL Light.,Pegasus Voyager,9665736,Suezmax,6,Arab Lt.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52506,2025-08-16 23:35:00,2025-09-23 21:18:00,37,25419839,Hafeet,9928009,VLCC+,"299,425.00","1,849,996.00",Juaymah,Angra dos Reis,Brazil,Latin America,False,None,None,<NA>,unknown,1,Arab Lt.
52467,2025-08-19 00:00:00,2025-08-28 08:18:00,9,25535725,C.Genuine,9929663,VLCC+,"300,419.00","1,933,361.00",Al Basrah,Jamnagar Refinery,India,South-Central Asia,False,None,None,<NA>,unknown,1,Basrah Hy.
52791,2025-08-01 15:39:00,2025-08-15 06:38:00,13,25429820,Seaways Endeavor,9933573,VLCC+,"299,365.00","650,001.00",Umm Lulu,Pengerang DIT,Malaysia,South-East Asia,False,None,None,<NA>,unknown,1,Umm Lulu
52228,2025-08-31 18:59:00,2025-09-27 08:59:00,26,25539440,Eagle Veracruz,9933626,VLCC+,"299,525.00","1,021,981.00",Al Basrah,Huizhou Refinery,China,Eastern Asia,False,None,None,<NA>,unknown,1,Basrah Med.


In [11]:
m_rep.to_csv('inspec_tankers_meg.csv')

## 6. Mother owners

In [12]:
own = owner_rank(df, top_n=25)
display(own)
fig = px.bar(
    own.head(15).sort_values("volume_bbl"),
    x="volume_bbl", y="vessel_owner", orientation="h",
    title=f"{cdef.label} [{FLOW_SCOPE}] — top mother owners",
)
fig.update_layout(height=480)
fig.show()

,vessel_owner,cargoes,volume_bbl,n_imo,share_vol
139,Mitsui,2363,"1,814,975,005.00",24,0.07
21,Bahri,1936,"1,693,440,388.00",37,0.06
36,COSCO,1571,"1,665,483,220.00",42,0.06
198,Sinokor,1583,"1,633,403,439.00",75,0.06
147,NYK Mitsubishi JV,1606,"1,115,445,572.00",18,0.04
144,NITC,826,"1,112,333,956.00",48,0.04
117,Kuwait Energy,745,"1,064,297,069.00",11,0.04
180,SCI,1017,"1,010,975,163.00",18,0.04
63,Dynacom,1180,"947,383,809.00",36,0.03
131,Maran Tankers Management Inc,1161,"932,060,921.00",45,0.03


## 7. Loading cadence

In [13]:
busy = loading_cadence(df).head(15)
display(busy)
wd = weekday_mix(df)
fig = px.bar(wd, x="weekday", y="cargoes",
             title=f"{cdef.label} [{FLOW_SCOPE}] — cargoes by weekday")
fig.update_layout(height=360)
fig.show()

,load_day,cargoes,volume_bbl
1922,2023-04-07,33,"28,349,373.00"
814,2020-03-25,32,"22,496,617.00"
115,2018-04-26,32,"18,816,186.00"
293,2018-10-21,31,"25,283,368.00"
167,2018-06-17,31,"23,323,131.00"
152,2018-06-02,30,"27,082,407.00"
673,2019-11-05,30,"23,925,389.00"
827,2020-04-07,30,"21,086,217.00"
458,2019-04-04,30,"22,438,123.00"
2914,2025-12-24,30,"27,333,916.00"


## 8. Snapshot + cross-corridor

In [14]:
summary = pd.Series({
    "corridor": cdef.key,
    "flow_scope": FLOW_SCOPE,
    "trades": df["trade_id"].nunique(),
    "mother_imos": df["imo"].nunique(),
    "daughter_imos": df["imo_2"].nunique(),
    "mbd_latest": m["mbd"].iloc[-1] if len(m) else None,
    "sts_vol_share": deliv.get("sts_vol_share"),
    "daughter_vol_share": deliv.get("daughter_vol_share"),
    "daughter_vlcc_vol_share": deliv.get("daughter_vlcc_vol_share"),
    "top10_mother_lift_share": conc.get("top10_lift_share") if len(conc) else None,
})
display(summary.to_frame("value"))

rows = []
for key in CORRIDORS:
    raw = load_crude_exports(con, key, status=STATUS, product=PRODUCT, start=START, end=END)
    d = apply_flow_scope(raw, FLOW_SCOPE)
    if d.empty:
        continue
    sd = sts_delivery_summary(d)
    cc = fleet_concentration(d)
    rows.append({
        "corridor": key,
        "flow_scope": FLOW_SCOPE,
        "trades": d["trade_id"].nunique(),
        "mbd_mean": monthly_export_mbd(d)["mbd"].mean(),
        "sts_vol_share": sd.get("sts_vol_share"),
        "daughter_vol_share": sd.get("daughter_vol_share"),
        "daughter_vlcc_vol_share": sd.get("daughter_vlcc_vol_share"),
        "n_daughters": sd.get("n_unique_daughters"),
        "top10_mother_lift_share": cc.get("top10_lift_share"),
        "intra_vol_share": flow_scope_summary(raw).set_index("flow_scope")["share_vol"].get("INTRA"),
    })
display(Markdown(f"### Cross-corridor (`FLOW_SCOPE={FLOW_SCOPE}`)"))
display(pd.DataFrame(rows))

,value
corridor,MEG
flow_scope,INTER
trades,47342
mother_imos,1898
daughter_imos,964
mbd_latest,1.61
sts_vol_share,0.08
daughter_vol_share,0.08
daughter_vlcc_vol_share,0.03
top10_mother_lift_share,0.03


### Cross-corridor (`FLOW_SCOPE=INTER`)

,corridor,flow_scope,trades,mbd_mean,sts_vol_share,daughter_vol_share,daughter_vlcc_vol_share,n_daughters,top10_mother_lift_share,intra_vol_share
0,BRAZIL,INTER,10498,1.54,0.81,0.81,0.61,912.00,0.29,0.53
1,WAF,INTER,16951,3.65,0.05,0.05,0.01,614.00,0.03,0.05
2,MEG,INTER,47342,13.67,0.08,0.08,0.03,964.00,0.03,0.04
3,USG,INTER,16157,3.16,0.28,0.28,0.25,"1,002.00",0.10,0.05
